In [1]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_parquet("data/processed/model_table.parquet")
y = df["default_flag"]
x = df.drop(columns=["id", "default_flag", "issue_year", "issue_date", "funded_amnt", "annual_inc"])
x = pd.get_dummies(x, columns=["home_ownership", "verification_status", "purpose"], drop_first=True, dtype=int)

print(f"Shape of x: {x.shape}")
print(f"Mean of y: {y.mean()}")

train_mask = df["issue_year"] <= 2014
y_train = y.loc[train_mask]
y_test = y.loc[~train_mask]

print(f"Rows in train set: {y_train.shape[0]}, Rows in test set: {y_test.shape[0]}")
print(f"Mean of y_train: {y_train.mean()}, Mean of y_test: {y_test.mean()}")

Shape of x: (545779, 80)
Mean of y: 0.14068331687367963
Rows in train set: 262992, Rows in test set: 282787
Mean of y_train: 0.1319165601995498, Mean of y_test: 0.14883640337073487


In [ ]:
df = pd.read_parquet("data/processed/model_table.parquet")

x = df.drop(columns=["id", "default_flag", "issue_year", "issue_date", "funded_amnt", "annual_inc"])
x = pd.get_dummies(
    x,
    columns=["home_ownership", "verification_status", "purpose"],
    drop_first=True,
    dtype=int,
)

prune_cols = [
    "num_sats", "num_rev_tl_bal_gt_0", "total_acc", "tot_hi_cred_lim",
    "total_il_high_credit_limit", "total_bc_limit", "bc_util",
    "ever_mths_since_last_major_derog", "ever_mths_since_recent_bc_dlq",
    "ever_mths_since_recent_revol_delinq", "ever_mths_since_last_delinq",
    "ever_mths_since_last_record", "pct_tl_nvr_dlq", "mo_sin_old_rev_tl_op",
    "revol_bal", "total_rev_hi_lim", "pub_rec", "num_op_rev_tl", "num_rev_accts",
    "mo_sin_rcnt_rev_tl_op", "mths_since_last_record", "mths_since_recent_revol_delinq",
    "num_tl_90g_dpd_24m",
]
x = x.drop(columns=prune_cols)

coefs = pd.read_csv("outputs/lr_coefficients.csv")
mask_2015 = df["issue_year"] == 2015
X = x.loc[mask_2015].copy()
X = X[coefs["feature"]]

print(f"2015 rows: {len(X)}")
print(f"Feature columns: {X.shape[1]}")
assert X.columns.tolist() == coefs["feature"].tolist()
print(f"Feature order matches coefficients: {len(X.columns)} features")

z = (X - coefs["mean"].values) / coefs["scale"].values
contrib = z * coefs["coef"].values

import json
with open("outputs/metrics.json") as f:
    intercept = json.load(f)["lr_57"]["intercept"]
pd_check = 1 / (1 + np.exp(-(intercept + contrib.sum(axis=1))))

scored_test = pd.read_parquet("data/processed/scored_test.parquet")
pd_comparison = pd.DataFrame({"id": df.loc[mask_2015, "id"].to_numpy(), "pd_check": pd_check.to_numpy()}).merge(
    scored_test[["id", "pd_lr"]],
    on="id",
    how="left",
    validate="one_to_one",
)
pd_comparison = pd_comparison.set_index("id").loc[df.loc[mask_2015, "id"]].reset_index()
max_abs_diff = (pd_comparison["pd_check"] - pd_comparison["pd_lr"]).abs().max()
print(max_abs_diff)

declined_mask = pd_comparison["pd_lr"].ge(0.18)
declined = pd_comparison.loc[declined_mask].copy()
top3 = contrib.loc[declined_mask.to_numpy()].apply(
    lambda row: row.nlargest(3).index.tolist(),
    axis=1,
)

print(f"Declined applicants: {len(declined)} ({len(declined) / len(pd_comparison)})")
print("Top 10 reason features:")
print(top3.explode().value_counts().head(10))

2015 rows: 282787
Feature columns: 57
Feature order matches coefficients: 57 features
2.55351295663786e-15
Declined applicants: 71,287 (25.2%)
Top 10 reason features:
fico                    38457
acc_open_past_24mths    33596
loan_to_income          24324
dti                     21226
log_annual_inc          20509
num_tl_op_past_12m      13971
home_ownership_RENT     11982
percent_bc_gt_75        11145
emp_length_missing       9757
inq_last_6mths           6816
Name: count, dtype: int64
